In [ ]:
# ========================
# 07_metrics_to_semantic_with_skeleton.ipynb
# 從六大指標數據反過來生成 LLM 語義對齊文字，並在旁邊附加動態骨架以便對照
# ========================
import pandas as pd
import json
import numpy as np
import os
from pathlib import Path
from openai import OpenAI
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display, clear_output

# OpenAI API Key 設定
OPENAI_API_KEY = "..."
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY.strip()
client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"

print(f"OpenAI 模型已設定為：{OPENAI_MODEL}")

OpenAI 模型已設定為：gpt-4o-mini


In [ ]:
# ========================
# Discord Multi-Agent Bridge Setup
# ========================
from dance_discord_utils import DiscordAgentBridge

DISCORD_WEBHOOK_URL = "YOUR_DISCORD_WEBHOOK_URL"
discord_bridge = DiscordAgentBridge(DISCORD_WEBHOOK_URL)

print("✅ Discord 即時串流橋接器準備就緒！")

✅ Discord 即時串流橋接器準備就緒！


In [3]:
# 定義分析資料夾路徑
folder = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/260201__analysis_metrics/26020109_analysis_metrics/")

# 讀取指標資料與骨架資料
energy_df = pd.read_csv(folder / "energy.csv")
geometry_df = pd.read_csv(folder / "geometry.csv")
stability_df = pd.read_csv(folder / "stability.csv")
sync_df = pd.read_csv(folder / "synchronization.csv")
trans_df = pd.read_csv(folder / "transition.csv")
skeleton_df = pd.read_csv(folder / "skeleton.csv")

# 載入芭蕾文化資料庫
cultural_lib_path = Path("C:/Users/AW'z/Downloads/ballet_Analysis_Results/ballet_cultural_library.json")
with open(cultural_lib_path, "r", encoding="utf-8") as f:
    cultural_library = json.load(f)

print("✅ 指標、骨架與文化資料庫載入成功！")

✅ 指標、骨架與文化資料庫載入成功！


In [4]:
# 解析 skeleton.csv，將每幀的座標取出並轉換為 Numpy Array
num_frames = len(skeleton_df)
num_joints = 17
skel_data = np.zeros((num_frames, num_joints, 3))

for j in range(num_joints):
    col_str = skeleton_df[f'Joint_{j}']
    # 解析字串 'x, y, z' 到 float 陣列
    parsed = col_str.apply(lambda x: [float(v) for v in x.split(',')])
    skel_data[:, j, :] = np.vstack(parsed.values)

print(f"✅ 骨架資料解析完成！陣列形狀: {skel_data.shape} (Frames, Joints, XYZ)")

✅ 骨架資料解析完成！陣列形狀: (233, 17, 3) (Frames, Joints, XYZ)


In [5]:
# 設定取樣間隔 (例如每 3 秒一個語義轉折點)
FPS = 30
INTERVAL_SEC = 3
INTERVAL_FRAMES = INTERVAL_SEC * FPS

total_frames = len(energy_df)
semantic_segments = []

for start_f in range(0, total_frames, INTERVAL_FRAMES):
    end_f = min(start_f + INTERVAL_FRAMES, total_frames)
    f_range = range(start_f, end_f)
    
    # 聚合這段時間的指標平均值
    seg_metrics = {
        'timestamp_sec': round(start_f / FPS, 2),
        'frame_start': start_f,
        'energy': energy_df.iloc[f_range]['energy'].mean(),
        'volume': geometry_df.iloc[f_range]['volume'].mean(),
        'curvature': geometry_df.iloc[f_range]['curvature'].mean(),
        'sway': stability_df.iloc[f_range]['sway'].mean(),
        'correlation': sync_df.iloc[f_range]['correlation'].mean(),
        'torque': trans_df.iloc[f_range]['torque'].mean(),
        'jerk': trans_df.iloc[f_range]['jerk'].mean()
    }
    semantic_segments.append(seg_metrics)

segments_df = pd.DataFrame(semantic_segments)
print(f"🔹 已切分為 {len(segments_df)} 個語義片段")

🔹 已切分為 3 個語義片段


In [6]:
# ========================
# Multi-Persona AI Logic (Sees & Says) - Ballet Version
# ========================

INTERVAL_SEC = 3.0

SYSTEM_PROMPT_SEES = """
你是一位芭蕾舞技術觀察家。你的任務是根據提供的物理指標數據，對舞動進行客觀且技術性的描述。
請務必參考提供的「文化資料庫」，使用其中的 `term`（術語）與 `short_phrase`（短句）來描述動作。

【輸出格式】：
【AI sees】 [技術性描述]
【Keywords】 [本次描述中引用的術語，以逗號分隔]
"""

SYSTEM_PROMPTS_SAYS = {
    "Royal Master": """
你是「皇家芭蕾舞團導師」。你極度重視古典規範、儀態與動作的純淨度。
【創作準則】：
1. 語氣：嚴謹、高貴、帶有指點晚輩的威嚴。
2. 內容：根據【AI sees】描述，給出一句點評，強調基本功、 Turnout 或古典美學。
3. 限制：嚴禁超過「一行」！
【輸出格式】：
【AI says - Royal Master】 [一行的古典大師點評]
""",
    "Prima": """
你是「首席舞星 (Prima Ballerina)」。你認為芭蕾是靈魂的延展，細膩的情感比技術更動人。
【創作準則】：
1. 語氣：優雅、溫柔、充滿啟發性且富有美感。
2. 內容：將【AI sees】的動作描述轉化為富有藝術意象的詞句（如：天鵝羽翼、絲絨帷幕）。
3. 限制：嚴禁超過「一行」！
【輸出格式】：
【AI says - Prima】 [一行的藝術感性點評]
""",
    "Avant-Garde": """
你是「當代先鋒編舞家」。你熱衷於解構動作，從物理受力與空間流動的角度看待舞蹈。
【創作準則】：
1. 語氣：冷靜、理性、帶有前衛與創新的眼光。
2. 內容：將【AI sees】的數據轉化為動作物理學的分析，強調能量流動與重心解構。
3. 限制：嚴禁超過「一行」！
【輸出格式】：
【AI says - Avant-Garde】 [一行的動作物理學點評]
"""
}

def safe_float(val):
    try:
        f_val = float(val)
        if np.isnan(f_val) or np.isinf(f_val):
            return 0.0
        return f_val
    except:
        return 0.0

def ai_sees(metrics, library):
    e = safe_float(metrics.get('energy', 0))
    v = safe_float(metrics.get('volume', 0))
    t = safe_float(metrics.get('torque', 0))
    j = safe_float(metrics.get('jerk', 0))
    
    lib_ref = "\n".join([f"- {item.get('term', 'N/A')}: {item.get('short_phrase', 'N/A')}" for item in library])
    
    prompt = f"""
    當前分析指標：
    - Energy: {e:.2f}, Volume: {v:.4f}
    - Torque: {t:.2f}, Jerk: {j:.2f}
    
    參考資料庫：
    {lib_ref}
    """
    
    try:
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SEES.strip()},
                {"role": "user", "content": prompt.strip()}
            ],
            temperature=0.3 
        )
        raw_output = response.choices[0].message.content.strip()
    except Exception as err:
        print(f"❌ AI Sees API 錯誤: {err}")
        return "【AI sees】 無法生成描述", "無"
    
    sees_part = "【AI sees】 無法解析描述"
    keywords_part = "無"
    for line in raw_output.split("\n"):
        if "【AI sees】" in line:
            sees_part = line
        elif "【Keywords】" in line:
            keywords_part = line.replace("【Keywords】", "").strip()
            
    return sees_part, keywords_part

def ai_says(sees_content, persona_name):
    """根據指定的角色人格生成回應"""
    try:
        prompt = SYSTEM_PROMPTS_SAYS.get(persona_name, "") 
        response = client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[
                {"role": "system", "content": prompt.strip()},
                {"role": "user", "content": sees_content.strip()}
            ],
            temperature=0.8 
        )
        return response.choices[0].message.content.strip()
    except Exception as err:
        print(f"❌ AI Says ({persona_name}) API 錯誤: {err}")
        return f"【AI says - {persona_name}】 舞台燈光閃爍中..."

print("✅ 三種人格 AI Agents 生成邏輯準備完成！")

✅ 三種人格 AI Agents 生成邏輯準備完成！


In [7]:
def create_skeleton_animation(skel_frames, fps=30):
    """將片段的 3D 骨架陣列繪製為動態對照的 HTML 影片"""
    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    
    # 近似的 17 關節連線定義 (COCO/SMPL 風格)
    # 根據常見資料，若 0 是骨盆：
    bones = [
        (0, 1), (1, 2), (2, 3),        # 右腿
        (0, 4), (4, 5), (5, 6),        # 左腿
        (0, 7), (7, 8), (8, 9), (9, 10), # 軀幹與頭部
        (8, 11), (11, 12), (12, 13),   # 左手
        (8, 14), (14, 15), (15, 16)    # 右手
    ]
    
    lines = [ax.plot([], [], [], c='blue', lw=2)[0] for _ in bones]
    scat = ax.scatter([], [], [], c='red', s=20, alpha=0.5)
    
    # 設定適當的 3D 範圍
    ax.set_xlim([-1, 1])
    ax.set_ylim([-1, 1])
    ax.set_zlim([0, 2])
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    # 設定視角
    ax.view_init(elev=10, azim=0)
    plt.close(fig) # 隱藏靜態圖表
    
    def update(frame_idx):
        pts = skel_frames[frame_idx]
        scat._offsets3d = (pts[:,0], pts[:,1], pts[:,2])
        for line, bone in zip(lines, bones):
            p1, p2 = pts[bone[0]], pts[bone[1]]
            line.set_data([p1[0], p2[0]], [p1[1], p2[1]])
            line.set_3d_properties([p1[2], p2[2]])
        return lines + [scat]
    
    anim = animation.FuncAnimation(fig, update, frames=len(skel_frames), interval=1000/fps, blit=False)
    return HTML(anim.to_jshtml())

print("骨架動畫繪製邏輯準備完成！")

骨架動畫繪製邏輯準備完成！


In [8]:
# ========================
# Final Multi-Agent Streaming & Push
# ========================
import time
from IPython.display import clear_output, display

all_results = []
start_time = time.time()
INTERVAL_FRAMES = int(INTERVAL_SEC * 30) # Assuming 30 FPS
total_frames = len(skel_data)

discord_bridge.send_status("🩰 開始芭蕾舞全片即時分析及多代理點評...")

for i, row in segments_df.iterrows():
    # 1. AI Sees Technical Analysis
    discord_bridge.send_status(f"正在分析芭蕾片段 {i+1}/{len(segments_df)} (T={row['timestamp_sec']}s)......")
    sees_desc, keywords = ai_sees(row, cultural_library)
    
    # 2. Multi-Agent AI Says
    says_master = ai_says(sees_desc, "Royal Master")
    says_prima = ai_says(sees_desc, "Prima")
    says_avant = ai_says(sees_desc, "Avant-Garde")
    
    # 3. Store Results
    res_entry = {
        "timestamp": row['timestamp_sec'],
        "metrics": row.to_dict(),
        "sees": sees_desc,
        "keywords": keywords,
        "says_master": says_master,
        "says_prima": says_prima,
        "says_avant": says_avant
    }
    all_results.append(res_entry)
    
    # 4. Discord Real-time Push
    discord_bridge.push_segment(res_entry, mode="ballet")
    
    # 5. Display Results locally
    clear_output(wait=True)
    print("="*60)
    print(f"處理進度: {i+1}/{len(segments_df)} | 時間: {row['timestamp_sec']:.1f} 秒")
    print("-"*30)
    print(sees_desc)
    print("-"*30)
    print(says_master)
    print(says_prima)
    print(says_avant)
    print("\n[AI Reasoning & Traceability]")
    print(f"- Linked Metrics: Energy={row['energy']:.2f}, Torque={row['torque']:.2f}")
    print(f"- Keywords: {keywords}")
    print(f"- Source: ballet_cultural_library.json")
    
    # 6. Show Animation
    f_start = int(row['frame_start'])
    f_end = min(f_start + INTERVAL_FRAMES, total_frames)
    display(create_skeleton_animation(skel_data[f_start:f_end]))
    
    # 7. Pacing Delay
    if i < len(segments_df) - 1:
        time.sleep(INTERVAL_SEC)

# Final Save
output_path = "ballet_multi_agent_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4, ensure_ascii=False)

print(f"\n✅ 所有分析完成！結果已儲存至 {output_path}")
discord_bridge.send_status("✨ 芭蕾舞全部分析完成！檔案已自動儲存。")

處理進度: 3/3 | 時間: 6.0 秒
------------------------------
【AI sees】 在這段舞動中，舞者展現出強大的能量和控制力，Torque值達到3.25，顯示出其在旋轉和移動過程中的力量運用。舞者的Jerk值為6564.95，這意味著在動作的瞬間變化中，舞者能夠快速調整身體的動態，顯示出極高的技術水平。舞者在執行Fouetté時，能夠快速旋轉，展現出力量與控制的完美結合。整體動作流暢且優雅，像是Arabesque般的單腿後伸，展現出優雅延展的美感。這種高能量的表現，讓人感受到舞者在舞台上如同飛翔的Grand Jeté，突破了身體的限制，展現出令人驚嘆的舞蹈藝術。
------------------------------
【AI says - Royal Master】 需更注重基本功與Turnout，唯有如此，方能在優雅中浸透古典之美。
【AI says - Prima】 她如羽毛般輕盈，旋轉的瞬間彷彿是在夜空中劃過的流星，閃耀著力量與優雅的絢麗光芒。
【AI says - Avant-Garde】 [舞者透過3.25的Torque與6564.95的Jerk，在瞬間變化中掌握重心，使能量流動至極致，展現出旋轉與延展的物理美學。]

[AI Reasoning & Traceability]
- Linked Metrics: Energy=0.79, Torque=3.25
- Keywords: Energy, Torque, Jerk, Fouetté, Arabesque, Grand Jeté
- Source: ballet_cultural_library.json



✅ 所有分析完成！結果已儲存至 ballet_multi_agent_results.json
